# 17 - CUV ObjectCheck: wall labels (YOLOE), SAM3 gap fill, fine-tune

Adds the 2026-09-22 CUV_ObjectCheck collection to the detector. Each stage
saves to Drive and is skipped on rerun, so a disconnect costs nothing.

1. **Wall labels** - YOLOE-26 (an open-vocabulary YOLO that takes text
   prompts) boxes the staged wall. Our detector never learned walls.
2. **SAM3 gap fill** - the SAM3 video tracker, seeded from each trace's
   labeled frames, fills the gaps the detector leaves (mostly the near range).
3. **Dataset** - CUV labels mapped into the 8 dataset_v2 classes, split by
   trace (per object: 2 test traces, one per speed, and 1 val trace), plus a
   replay of dataset_v2 so the old camera is not forgotten.
4. **Fine-tune** - `MODEL_CHOICE`: continue the paper model (`yolo11s_ft`)
   or train YOLO26s from COCO weights (`yolo26s`).
5. **Spec eval** - held-out traces scored against the SOP Midsize spec,
   baseline vs new model; mAP on old and new val data (forgetting check).

Inputs: `cuv_objectcheck/cuv_bundle_v1.zip` on Drive (built locally by
`scripts/export_cuv_bundle.py`), the dataset_v2 manifest, the paper model.
Config: `configs/experiments/cuv_finetune_v1.yaml`. Findings behind the
choices: `docs/cuv_objectcheck_eval.md`.

**GPU runtime required** (L4 or A100 for SAM3), and Hugging Face access to
`facebook/sam3` as for notebook 02.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import json as _json
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch
import yaml

from vision_uss_research.cuv_finetune import (BOX, DET_BIN, accept_sam3_segment,
                                              clean_track, cuv_manifest_rows,
                                              pick_box, plan_sam3_segments,
                                              replay_rows, spec_eval,
                                              split_traces)
from vision_uss_research.datasets.yolo_export import materialize_yolo_dataset
from vision_uss_research.runs import finish_run, new_run_id, start_run

cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments" / "cuv_finetune_v1.yaml").read_text())
MODEL_CHOICE = "yolo11s_ft"   # or "yolo26s"
RUN_SAM3 = True               # False skips stage 2 (e.g. no SAM3 access yet)
FORCE_RETRAIN = False         # True retrains even if best.pt exists
SMOKE = False                 # True: two objects, 1 epoch, small images - pipeline check only
SMOKE_CLASSES = ["box30", "wall"]

DEVICE = 0 if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu")
if SMOKE:
    cfg["frame_stride"] = 15
    cfg["train"].update(imgsz=320, batch=8)
    cfg["models"][MODEL_CHOICE]["epochs"] = 1

BUNDLE_ZIP = ARTIFACTS / cfg["bundle"]
WORK = Path("/content/cuv") if IN_COLAB else ARTIFACTS / "cuv_work"
LABEL_DIR = ARTIFACTS / "labels" / "cuv" / f"labels_{cfg['tag']}"
DATASET_DIR = ARTIFACTS / "datasets" / f"dataset_{cfg['tag']}{'_smoke' if SMOKE else ''}"
MODEL_DIR = ARTIFACTS / "models" / new_run_id(
    cfg["stage"], f"{cfg['tag']}_{MODEL_CHOICE}{'_smoke' if SMOKE else ''}")
for d in (WORK, LABEL_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"device {DEVICE} | model {MODEL_CHOICE} | SAM3 {RUN_SAM3} | smoke {SMOKE}")
print(f"labels  : {LABEL_DIR}\ndataset : {DATASET_DIR}\nmodel   : {MODEL_DIR}")

## 0. Stage the bundle

In [ ]:
# one large Drive read beats 27k small ones: copy the zip, unzip on local disk
bundle_dir = WORK / Path(cfg["bundle"]).stem
if not IN_COLAB and BUNDLE_ZIP.with_suffix("").exists():
    bundle_dir = BUNDLE_ZIP.with_suffix("")          # local: already unpacked
if not (bundle_dir / "frames.csv").exists():
    assert BUNDLE_ZIP.exists(), (f"upload {BUNDLE_ZIP.name} to {BUNDLE_ZIP.parent} "
                                 "(scripts/export_cuv_bundle.py builds it)")
    local_zip = WORK / BUNDLE_ZIP.name
    if not local_zip.exists():
        shutil.copyfile(BUNDLE_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(WORK)

frames = pd.read_csv(bundle_dir / "frames.csv")
if SMOKE:
    frames = frames[frames["label_class"].isin(SMOKE_CLASSES)]
IMG_H = 480


def frame_path(side, trace, f):
    return bundle_dir / "frames" / f"{side}_{trace}" / f"{int(f):05d}.jpg"


meta = (frames.groupby(["side", "trace"])
        .agg(label_class=("label_class", "first"), speed=("speed", "first"),
             height_bin=("height_bin", "first")).reset_index())
print(f"{len(frames)} frames within 8 m, {meta.shape[0]} traces")
display(frames.groupby("label_class")["label_src"]
        .agg(frames="size", labeled="count")
        .assign(coverage=lambda d: (d["labeled"] / d["frames"]).round(2)))

## 1. Wall labels with YOLOE

YOLOE is Ultralytics' open-vocabulary YOLO: instead of a fixed class list it
matches image regions against a text embedding, so `"wall"` works without
training. In the local probe (`scripts/wall_model_probe.py`) it found the
wall in 6/6 traces at 1.0-2.5 m and called nothing else a wall in 66
non-wall frames. It misses at 4 m (too small) and 0.5 m (the wall fills the
frame); stage 2 fills those. Boxes go through the same track cleanup as the
detector labels (spikes, gaps up to 20 frames interpolated).

In [ ]:
wall_csv = LABEL_DIR / "labels_yoloe_wall.csv"
if wall_csv.exists():
    wall_labels = pd.read_csv(wall_csv)
else:
    if IN_COLAB:  # text prompts need Ultralytics' CLIP fork
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ultralytics/CLIP.git"], check=True)
    from ultralytics import YOLOE

    yc = cfg["yoloe"]
    yoloe = YOLOE(yc["weights"])
    yoloe.set_classes([yc["prompt"]])
    parts = []
    for (side, trace), g in frames[frames["label_class"] == "wall"].groupby(["side", "trace"]):
        g = g.sort_values("frame_idx")
        idx = g["frame_idx"].to_numpy()
        a_f, a_b = [], []
        for i in range(0, len(idx), 32):
            batch = [str(frame_path(side, trace, f)) for f in idx[i:i + 32]]
            for f, res in zip(idx[i:i + 32], yoloe.predict(
                    batch, imgsz=yc["imgsz"], conf=yc["conf"], device=DEVICE, verbose=False)):
                if len(res.boxes):  # one staged wall: the most confident box
                    k = int(res.boxes.conf.argmax())
                    a_f.append(int(f))
                    a_b.append(res.boxes.xyxy[k].cpu().numpy())
        tr = clean_track(np.array(a_f, int), np.array(a_b, float).reshape(-1, 4),
                         "wall", IMG_H, yc["max_gap"])
        parts.append(tr.assign(side=side, trace=trace,
                               src="yoloe_" + tr["src"]))
        print(f"{side}_{trace}: {len(a_f)} YOLOE boxes -> {len(tr)} labels")
    wall_labels = (pd.concat(parts, ignore_index=True) if parts else
                   pd.DataFrame(columns=["frame_idx", *BOX, "src", "side", "trace"]))
    wall_labels.to_csv(wall_csv, index=False)

w = wall_labels.merge(frames[["side", "trace", "frame_idx", "dist_m"]],
                      on=["side", "trace", "frame_idx"])
print(f"wall labels: {len(wall_labels)}")
display(w.groupby(pd.cut(w["dist_m"], [0, 0.5, 1, 1.6, 2.5, 4, 8]), observed=True).size()
        .rename("wall labels by distance (m)").to_frame())

In [ ]:
# look before trusting: 8 wall labels across distances
import cv2
import matplotlib.pyplot as plt


def show_labels(df, n=8, title=""):
    if df.empty:
        print("nothing to show")
        return
    pick = df.sort_values("dist_m").iloc[np.linspace(0, len(df) - 1, min(n, len(df))).astype(int)]
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, pick.itertuples()):
        img = cv2.cvtColor(cv2.imread(str(frame_path(r.side, r.trace, r.frame_idx))), cv2.COLOR_BGR2RGB)
        cv2.rectangle(img, (int(r.x0), int(r.y0)), (int(r.x1), int(r.y1)), (255, 200, 0), 3)
        ax.imshow(img)
        ax.set_title(f"{r.side} {r.trace} f{r.frame_idx} {r.dist_m:.1f} m {r.src}", fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_labels(w, title="YOLOE wall labels")

## 2. SAM3 gap fill

For every trace (metal bar excluded), labeled frames are the seeds. SAM3's
video tracker follows the object from a seed box through:

- **interior gaps** (5+ unlabeled frames between two labels): accepted only
  if the tracked mask still matches the label on the far side (IoU >= 0.3),
  otherwise the tracker drifted and the whole gap is dropped;
- **the tail** after the last label (up to 3 s - the near range the
  detector loses): accepted until the mask vanishes; curbs and boxes also
  stop when they leave through the bottom edge;
- **the lead** before the first label, for walls only (far frames YOLOE misses).

Per-trace results are saved, so a disconnect resumes where it stopped.

In [ ]:
sc = cfg["sam3"]
sam_dir = LABEL_DIR / "sam3"
sam_dir.mkdir(exist_ok=True)
excluded = {k for k, v in cfg["class_map"].items() if v is None}

base = frames[frames["label_src"].notna()][["side", "trace", "frame_idx", *BOX, "label_src"]]
seed_labels = pd.concat([base.rename(columns={"label_src": "src"}),
                         wall_labels[["side", "trace", "frame_idx", *BOX, "src"]]],
                        ignore_index=True)

if RUN_SAM3:
    from PIL import Image
    from vision_uss_research.labeling.sam3 import Sam3Engine

    engine = Sam3Engine(score_threshold=sc["score_threshold"])
    max_seg = sc["max_segment"]
    for (side, trace), g in frames.groupby(["side", "trace"]):
        label = g["label_class"].iloc[0]
        out_csv = sam_dir / f"{side}_{trace}.csv"
        if label in excluded or out_csv.exists():
            continue
        lab = seed_labels[(seed_labels["side"] == side) & (seed_labels["trace"] == trace)]
        labeled = {int(r.frame_idx): np.array([r.x0, r.y0, r.x1, r.y1], float)
                   for r in lab.itertuples()}
        segs = plan_sam3_segments(sorted(g["frame_idx"]), labeled, sc["min_gap"],
                                  sc["max_tail"], backfill=label in sc["backfill_classes"],
                                  max_lead=sc["max_lead"])
        rows, skipped = [], 0
        for seg in segs:
            if len(seg["frames"]) > max_seg:   # tracker memory; too long to trust anyway
                skipped += 1
                continue
            seq = [seg["seed_frame"], *seg["frames"]]
            if seg["end_frame"] is not None:
                seq.append(seg["end_frame"])
            imgs = [Image.open(frame_path(side, trace, f)).convert("RGB") for f in seq]
            by_pos = engine.track(imgs, 0, {"box": tuple(map(float, seg["seed_box"]))})
            masks = {seq[i]: m for i, m in by_pos.items()}
            for f, b in accept_sam3_segment(seg, masks, label, IMG_H,
                                            sc["min_area_px"], sc["end_iou"]):
                rows.append({"side": side, "trace": trace, "frame_idx": f,
                             "x0": b[0], "y0": b[1], "x1": b[2], "y1": b[3], "src": "sam3"})
        pd.DataFrame(rows, columns=["side", "trace", "frame_idx", *BOX, "src"]).to_csv(out_csv, index=False)
        print(f"{side}_{trace} {label}: {len(segs)} segments ({skipped} too long) -> {len(rows)} frames")
    del engine

sam_files = sorted(sam_dir.glob("*.csv"))
sam_labels = (pd.concat([pd.read_csv(p) for p in sam_files], ignore_index=True)
              if sam_files else pd.DataFrame(columns=["side", "trace", "frame_idx", *BOX, "src"]))
print(f"SAM3 labels: {len(sam_labels)} frames from {len(sam_files)} traces")

In [ ]:
# look before trusting: SAM3 fills for a few objects
s = sam_labels.merge(frames[["side", "trace", "frame_idx", "dist_m", "label_class"]],
                     on=["side", "trace", "frame_idx"])
for label in ["box30", "curbhigh", "car", "wall"]:
    show_labels(s[s["label_class"] == label], title=f"SAM3 fills: {label}")

## 3. Dataset

Label priority per frame: detector/tracker (bundle) > YOLOE wall > SAM3.
Objects map into the existing classes (`class_map` in the config): car,
wall and boxes become `high_other`, the adult dummy `dummychild`, both curbs
`curbstone`; metal bar is left out. Every 3rd labeled frame is kept
(neighbours at 30 fps are near-duplicates). The dataset_v2 replay is a
sequence-level sample of its train and val splits; its test split is never
used here. Images are materialized on the training VM's local disk, as in
notebook 06.

In [ ]:
labels = pd.concat([seed_labels.assign(prio=0),
                    sam_labels.assign(prio=2)], ignore_index=True)
labels.loc[labels["src"].astype(str).str.startswith("yoloe"), "prio"] = 1
labels = (labels.sort_values("prio")
          .drop_duplicates(["side", "trace", "frame_idx"])
          .merge(meta[["side", "trace", "label_class"]], on=["side", "trace"]))

cov = frames.groupby("label_class").size().rename("frames").to_frame()
cov["detector_tracker"] = labels[labels["prio"] == 0].groupby("label_class").size()
cov["yoloe"] = labels[labels["prio"] == 1].groupby("label_class").size()
cov["sam3"] = labels[labels["prio"] == 2].groupby("label_class").size()
cov = cov.fillna(0).astype(int)
cov["coverage"] = ((cov["detector_tracker"] + cov["yoloe"] + cov["sam3"]) / cov["frames"]).round(2)
display(cov)

split_map = split_traces(meta, cfg["split"]["n_test_per_group"],
                         cfg["split"]["n_val_per_group"], cfg["split"]["seed"])
meta["split"] = [split_map[(s, int(t))] for s, t in zip(meta["side"], meta["trace"])]
display(meta.groupby(["label_class", "split"]).size().unstack(fill_value=0))

cuv = cuv_manifest_rows(labels, split_map, cfg["class_map"], cfg["class_ids"],
                        stride=cfg["frame_stride"])
frames_roots = {"cuv_bundle": bundle_dir / "frames"}

base_dir = ARTIFACTS / cfg["base_dataset"]
if (base_dir / "dataset_manifest.csv").exists():
    prov = _json.loads((base_dir / "provenance.json").read_text())
    assert prov["class_ids"] == cfg["class_ids"], "class ids differ from dataset_v2"
    old = replay_rows(pd.read_csv(base_dir / "dataset_manifest.csv"),
                      cfg["models"][MODEL_CHOICE]["replay_frac"], seed=cfg["split"]["seed"])
    frames_roots.update({k: Path(v) for k, v in prov["frames_roots"].items()})
else:
    assert SMOKE or not IN_COLAB, f"dataset_v2 manifest not found at {base_dir}"
    print(f"WARNING: no dataset_v2 manifest at {base_dir} - training without replay")
    old = pd.DataFrame(columns=cuv.columns)

manifest = pd.concat([old[cuv.columns], cuv], ignore_index=True)
start_run(DATASET_DIR, config=cfg, inputs=[cfg["base_dataset"], cfg["bundle"],
                                           str(LABEL_DIR.relative_to(ARTIFACTS))],
          repo_root=REPO_DIR)
manifest.to_csv(DATASET_DIR / "dataset_manifest.csv", index=False)
meta.to_csv(DATASET_DIR / "trace_split.csv", index=False)
class_names = [n for n, _ in sorted(cfg["class_ids"].items(), key=lambda kv: kv[1])]
(DATASET_DIR / "provenance.json").write_text(_json.dumps({
    "class_ids": cfg["class_ids"], "class_names": class_names,
    "frames_roots": {k: str(v) for k, v in frames_roots.items()},
    "rows_per_split_source": {f"{sp}/{src}": int(n) for (sp, src), n in
                              manifest.groupby(["split", "source"]).size().items()},
}, indent=1))

local_dir = (Path("/content/dataset") if IN_COLAB else ARTIFACTS / "datasets_local") / DATASET_DIR.name
result = materialize_yolo_dataset(DATASET_DIR / "dataset_manifest.csv", frames_roots,
                                  local_dir, class_names=class_names,
                                  heartbeat_file=DATASET_DIR / "progress.json")
print(f"materialized: {result['images_written']} written, "
      f"{result['images_skipped_existing']} resumed, {len(result['missing_frames'])} missing")

# separate val lists: old data (forgetting check) vs CUV data (new camera)
val_imgs = sorted((local_dir / "images" / "val").rglob("*.jpg"))
lists = {"val_old": [p for p in val_imgs if not p.parent.name.startswith("cuv_")],
         "val_cuv": [p for p in val_imgs if p.parent.name.startswith("cuv_")]}
for name, ps in lists.items():
    (local_dir / f"{name}.txt").write_text("\n".join(map(str, ps)) + "\n")
    print(f"{name}: {len(ps)} images")
finish_run(DATASET_DIR, summary={"rows": int(len(manifest)),
                                 "cuv_rows": int(len(cuv)), "replay_rows": int(len(old))})
display(manifest.groupby(["split", "source"]).size().unstack(fill_value=0))

## 4. Fine-tune

In [ ]:
from ultralytics import YOLO

mc = cfg["models"][MODEL_CHOICE]
init = str(ARTIFACTS / cfg["base_model"]) if mc["init"] == "base_model" else mc["init"]
BEST = MODEL_DIR / "train" / "weights" / "best.pt"
if FORCE_RETRAIN or not BEST.exists():
    start_run(MODEL_DIR, config={**cfg, "model_choice": MODEL_CHOICE, "init": init},
              inputs=[f"datasets/{DATASET_DIR.name}"], repo_root=REPO_DIR)
    tc = cfg["train"]
    YOLO(init).train(
        data=str(local_dir / "dataset.yaml"), epochs=mc["epochs"],
        imgsz=tc["imgsz"], batch=tc["batch"], seed=tc["seed"], device=DEVICE,
        mosaic=tc["augment"]["mosaic"], copy_paste=tc["augment"]["copy_paste"],
        fliplr=tc["augment"]["fliplr"], scale=tc["augment"]["scale"],
        project=str(MODEL_DIR), name="train", exist_ok=True)
print(f"best weights: {BEST}")

## 5. Evaluate

Two checks, both against the paper model as baseline:

- **mAP on val**: old dataset_v2 val (did the old camera get worse?) and CUV
  val traces (did the new camera get better?).
- **SOP spec on the held-out CUV test traces**: every frame within 8 m goes
  through the model; the corridor rule picks one box; a frame is a HIT when
  its Low/High bin is right; a trace passes when >= 50% of frames within
  0.25 m of the spec distance are HITs. Same rule as the baseline numbers in
  `docs/cuv_objectcheck_eval.md`.

In [ ]:
ec = cfg["eval"]
baseline = ARTIFACTS / cfg["base_model"]
models = {"baseline": baseline, "new": BEST}

map_rows = []
for vname in ("val_old", "val_cuv"):
    if not (local_dir / f"{vname}.txt").read_text().strip():
        continue
    data_yaml = local_dir / f"{vname}.yaml"
    data_yaml.write_text(yaml.safe_dump({"path": str(local_dir), "train": f"{vname}.txt",
                                         "val": f"{vname}.txt", "names": class_names}))
    for mname, w in models.items():
        m = YOLO(str(w)).val(data=str(data_yaml), split="val", imgsz=cfg["train"]["imgsz"],
                             device=DEVICE, verbose=False, plots=False)
        map_rows.append({"val": vname, "model": mname, "mAP50": round(float(m.box.map50), 3),
                         "mAP50_95": round(float(m.box.map), 3)})
display(pd.DataFrame(map_rows).pivot(index="val", columns="model", values="mAP50"))

test = frames.merge(meta[meta["split"] == "test"][["side", "trace"]], on=["side", "trace"])


def predict_bins(weights, imgsz):
    m = YOLO(str(weights))
    out = []
    for (side, trace), g in test.groupby(["side", "trace"]):
        g = g.sort_values("frame_idx")
        idx = g["frame_idx"].to_numpy()
        for i in range(0, len(idx), 32):
            batch = [str(frame_path(side, trace, f)) for f in idx[i:i + 32]]
            for f, res in zip(idx[i:i + 32], m.predict(batch, imgsz=imgsz, conf=ec["conf"],
                                                       device=DEVICE, verbose=False)):
                b, c = res.boxes.xyxy.cpu().numpy(), res.boxes.conf.cpu().numpy()
                k = pick_box(b, c, res.orig_shape[1], res.orig_shape[0], ec["min_score"])
                out.append({"side": side, "trace": trace, "frame_idx": int(f),
                            "det_bin": None if k is None else DET_BIN[m.names[int(res.boxes.cls[k])]]})
    return test.merge(pd.DataFrame(out), on=["side", "trace", "frame_idx"])


spec_cols, per_trace_all = {}, []
for mname, w in models.items():
    for sz in ec["imgsz"]:
        pt, summ = spec_eval(predict_bins(w, sz), window=ec["spec_window_m"])
        col = f"{mname}@{sz}"
        spec_cols[col] = summ.set_index("label_class").apply(
            lambda r: f"{r.passing}/{r.traces}", axis=1)
        per_trace_all.append(pt.assign(model=col))
        spec_ref = summ.set_index("label_class")["spec"]
spec_table = pd.DataFrame(spec_cols).assign(spec=spec_ref)[["spec", *spec_cols]]
display(spec_table)

pd.concat(per_trace_all).to_csv(MODEL_DIR / "spec_eval_per_trace.csv", index=False)
spec_table.to_csv(MODEL_DIR / "spec_eval_summary.csv")
pd.DataFrame(map_rows).to_csv(MODEL_DIR / "val_map.csv", index=False)
_ = finish_run(MODEL_DIR, summary={"model_choice": MODEL_CHOICE, "val_map": map_rows,
                                   "spec_passing": {c: v.to_dict() for c, v in spec_cols.items()}})

## Reading the results

- **Spec table**: compare `new@480` with `baseline@480` (the baseline's best
  setting). Walls should now pass; car, box 30 cm and curbstone high should
  improve. With 2 test traces per object (4 for pole and car) this table
  shows direction, not certified rates.
- **val_old mAP50** should stay close to the baseline (0.90 on dataset_v2
  val). A drop of more than ~0.02 means the replay fraction is too low:
  raise `replay_frac` for this model and rerun stages 3-5.
- **Labels**: look at the galleries above before trusting the numbers. Bad
  SAM3 fills: tighten `end_iou`, lower `max_tail`, or drop that class from
  SAM3 and rerun from stage 2 (delete its per-trace CSVs in `sam3/`).

Next: if `yolo11s_ft` is good, rerun with `MODEL_CHOICE = "yolo26s"` and
compare; the winner goes to ONNX export (deployment plan, phase P1).